# 01. Análise Exploratória: Indicadores Macroeconómicos
**Projeto:** Modelação de episódios recorrentes de stress financeiro em PME

Nesta fase da investigação, procedemos à caracterização do ambiente macroeconómico de Portugal (2005-2023). Estes indicadores representam as **variáveis ambientais** que, de acordo com a nossa fundamentação teórica, exercem uma pressão sistémica sobre a saúde financeira das PME, influenciando tanto a probabilidade de falência como a capacidade de recuperação (recorrência).

In [ ]:
import pandas as pd 
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.ticker import MaxNLocator

sns.set_theme(style="whitegrid")
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 12

macro_path = "../data/raw/macro/"
files = [f for f in os.listdir(macro_path) if f.endswith('.csv')]

# Mapping nomenclature
title_mapping = {
    'deaths_births_survivors_firms': 'Demografia Empresarial (Nascimentos vs Mortes)',
    'inflation_rate_goods_services': 'Taxa de Inflação (%)',
    'interest_rate_on_firm_loans_anual_mean': 'Taxa de Juro (PME < 1M€) (%)',
    'minmum_wage': 'Salário Mínimo Nacional (Euros)',
    'pib_by_year': 'PIB (Preços Correntes - M€)',
    'pib_capita': 'PIB per Capita (Euros)',
    'pib_growth': 'Crescimento Real do PIB (%)',
    'public_debt_%_pib': 'Dívida Pública (% do PIB)',
    'unemployment_rate': 'Desemprego (Total Inscritos - N.º)'
}

print(f"Foram identificados {len(files)} indicadores macroeconómicos para análise.")

## 1. Triagem e Inspeção de Dados Brutos

Dada a proveniência heterogénea dos dados (PORDATA, INE), esta secção valida a granularidade e a estrutura de filtros de cada ficheiro. O objetivo é garantir que extraímos séries temporais únicas e consistentes a nível nacional, eliminando ruído de metadados ou agregações regionais irrelevantes para o estudo.

In [ ]:
for file in files:
    df_tmp = pd.read_csv(os.path.join(macro_path, file), encoding='utf-8-sig', low_memory=False)
    print(f"\n{'='*60}")
    print(f"FICHEIRO: {file}")
    print(f"{'='*60}")
    
    print(f"Colunas identificadas: {list(df_tmp.columns)}")
    
    # Temporal integrity check (duplicate records per year)
    year_col = df_tmp.columns[0]
    if 'Ano' in year_col or 'Year' in year_col:
        counts = df_tmp.groupby(year_col).size()
        max_rows = counts.max()
        print(f"Máximo de registos por ano: {max_rows} (Requer filtragem se > 1)")
    
    # Identification of filter columns (PORDATA standard)
    filter_cols = [c for c in df_tmp.columns if any(x in c for x in ['02.', '03.', '04.', '05.', '06.'])]
    if filter_cols:
        print("\nValores únicos nos filtros:")
        for col in filter_cols:
            uniques = df_tmp[col].dropna().unique()
            print(f"  - {col}: {list(uniques[:3])}... ({len(uniques)} total)")
    
    print("\nPré-visualização (2 linhas):")
    display(df_tmp.head(2))

## 2. Harmonização e Limpeza Automatizada

Implementamos uma lógica de limpeza cirúrgica para converter as tabelas multidimensionais em séries temporais simples. Esta padronização é fundamental para a futura integração (merge) com o dataset de microdados ao nível do NIF-Ano.

In [ ]:
def clean_macro_df(file_name, df):
    key = file_name.replace('.csv', '')
    
    if 'deaths_births' in file_name:
        df = df[['Year', 'Births', 'Deaths']].copy()
        df.columns = ['year', 'firm_births', 'firm_deaths']
        return df.set_index('year')
    
    year_col = df.columns[0]
    value_col = [c for c in df.columns if 'Valor' in c][0]
    
    # Indicator-specific filtering logic
    if 'unemployment_rate' in file_name:
        if len(df.columns) > 2:
            df = df[(df.iloc[:, 1] == 'Portugal')]
            for i in range(2, len(df.columns) - 1):
                if 'Total' in df.iloc[:, i].unique():
                    df = df[df.iloc[:, i] == 'Total']
        
    elif 'inflation_rate' in file_name:
        df = df[df.iloc[:, 2] == 'Total']
    elif 'interest_rate' in file_name:
        df = df[df.iloc[:, 2].str.contains('Até 1 milhão', na=False, case=False)]
    elif 'minmum_wage' in file_name:
        df = df[df.iloc[:, 2].str.contains('Portugal continental', na=False, case=False)]
    elif 'pib_' in file_name or 'public_debt' in file_name:
        mask = df.iloc[:, 1:3].apply(lambda x: x.astype(str).str.contains('Portugal', na=False)).any(axis=1)
        df = df[mask]

    new_df = df[[year_col, value_col]].copy()
    new_df.columns = ['year', key]
    new_df['year'] = pd.to_numeric(new_df['year'], errors='coerce')
    new_df[key] = pd.to_numeric(new_df[key], errors='coerce')
    
    return new_df.dropna().sort_values('year').groupby('year').mean()

cleaned_dfs = []
for file in files:
    df_raw = pd.read_csv(os.path.join(macro_path, file), encoding='utf-8-sig', low_memory=False)
    df_clean = clean_macro_df(file, df_raw)
    cleaned_dfs.append(df_clean)
    print(f"Processado: {file} | Período: {df_clean.index.min()} - {df_clean.index.max()}")

## 3. Análise Visual de Séries Temporais
Validamos o comportamento dos indicadores face aos períodos de choque económico documentados: a **Intervenção da Troika (2011-2014)** e a pandemia **COVID-19 (2020-2021)**. Estes eventos são cruciais para validar a sensibilidade do nosso futuro modelo de sobrevivência a variáveis exógenas.

In [24]:
n_indicators = len(cleaned_dfs)
fig, axes = plt.subplots(nrows=(n_indicators + 1) // 2, ncols=2, figsize=(16, 5 * ((n_indicators + 1) // 2)))
axes = axes.flatten()

crises = [
    {'range': (2011, 2014), 'label': 'Crise Troika', 'color': 'red'},
    {'range': (2020, 2021), 'label': 'Pandemia COVID-19', 'color': 'orange'}
]

for i, df in enumerate(cleaned_dfs):
    ax = axes[i]
    key = df.columns[0] if len(df.columns) == 1 else "deaths_births_survivors_firms"
    pretty_title = title_mapping.get(key, key)
    
    if key == "deaths_births_survivors_firms":
        df.plot(kind='bar', ax=ax, width=0.8, color=['#27ae60', '#e74c3c'], alpha=0.8)
        ax.set_xticklabels(df.index.astype(int), rotation=45)
    else:
        sns.lineplot(data=df, x=df.index, y=key, ax=ax, marker='o', linewidth=2.5, color='#2c3e50')
    
    for crisis in crises:
        ax.axvspan(crisis['range'][0], crisis['range'][1], color=crisis['color'], alpha=0.1, 
                   label=crisis['label'] if i == 0 else "")
    
    ax.set_title(pretty_title, fontweight='bold', pad=15)
    ax.set_xlabel("Ano", fontsize=10)
    ax.set_ylabel("Valor", fontsize=10)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    
    if key != "deaths_births_survivors_firms":
        ax.set_xlim(2005, 2024)
    
    if i == 0: ax.legend(loc='upper left', frameon=True)

plt.tight_layout()
plt.show()

## 4. Dinâmicas de Correlação: Pearson vs. Spearman

Nesta análise, comparamos a correlação linear (Pearson) com a correlação baseada em postos (Spearman). Esta distinção é vital para detetar relações não-lineares e a robustez dos indicadores perante valores extremos (outliers) causados pelas crises financeiras.

In [ ]:
macro_final = pd.concat(cleaned_dfs, axis=1).sort_index().loc[2005:]

print("--- Estatística Descritiva (2005-2023) ---")
display(macro_final.describe().T)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 10))

# Pearson correlation
pearson_corr = macro_final.corr(method='pearson')
sns.heatmap(pearson_corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=1, ax=ax1)
ax1.set_title("Correlação de Pearson (Linear)", fontsize=18, fontweight='bold', pad=20)

# Spearman correlation
spearman_corr = macro_final.corr(method='spearman')
sns.heatmap(spearman_corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=1, ax=ax2)
ax2.set_title("Correlação de Spearman (Não-Linear/Robust)", fontsize=18, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

### Interpretação das Dependências Estruturais

A divergência entre as correlações revela a complexidade do ecossistema macroeconómico português. Por exemplo, a relação entre o **PIB per Capita** e as **Taxas de Juro** apresenta uma correlação de Spearman significativamente mais forte do que a de Pearson. 

Esta discrepância indica que, embora exista uma relação inversa estrutural (com o aumento da riqueza nacional, o risco de crédito e as taxas tendem a baixar), esta relação não é estritamente linear, sendo distorcida por choques exógenos e intervenções políticas (e.g., as medidas do BCE durante a pandemia). Estes padrões não-lineares reforçam a pertinência de utilizar modelos de Machine Learning como **Random Survival Forests (RSF)** e **DeepHit**, que possuem uma capacidade superior para capturar estes sinais ambientais complexos em comparação com o modelo de Cox tradicional.

## 5. Consolidação e Exportação
Os dados limpos são persistidos na camada 'interim', prontos para a integração com os microdados das PME provenientes do SABI.

In [26]:
output_file = '../data/interim/macro_consolidated.csv'
os.makedirs('../data/interim', exist_ok=True)
macro_final.to_csv(output_file)
print(f"\nSUCESSO: {macro_final.shape[1]} indicadores consolidados para um período de {len(macro_final)} anos.")
print(f"Ficheiro guardado em: {output_file}")

## 6. Dicionário de Variáveis Macroeconómicas (Codebook)

Esta tabela serve de referência para a dissertação, detalhando o significado técnico e a unidade de medida de cada variável ambiental utilizada nos modelos de sobrevivência.

| Variável | Descrição Técnica | Unidade | Fonte |
| :--- | :--- | :--- | :--- |
| `firm_births` | Número total de constituições de sociedades comerciais no ano. | N.º Absoluto | INE |
| `firm_deaths` | Número total de dissoluções de sociedades comerciais no ano. | N.º Absoluto | INE |
| `inflation_rate` | Variação média anual do Índice de Preços no Consumidor (IPC). | % | INE |
| `interest_rate` | Taxa de juro média anual para novos empréstimos a PME (montantes < 1M€). | % | INE |
| `minmum_wage` | Retribuição Mínima Mensal Garantida (RMMG) em vigor no ano civil. | EUR | INE |
| `pib_by_year` | Produto Interno Bruto a preços correntes (ótica da despesa). | MilhõesEUR | PORDATA |
| `pib_capita` | Rácio do PIB pela população residente média de Portugal. | EUR | PORDATA |
| `pib_growth` | Taxa de variação real (em volume) do Produto Interno Bruto. | % | PORDATA |
| `public_debt` | Dívida bruta consolidada das Administrações Públicas em % do PIB. | % | PORDATA |
| `unemployment_rate`| Média anual de desempregados inscritos nos centros de emprego. | N.º Absoluto | PORDATA |